# 🧪 GIADA Task 1 — Gate `m` di `Ca_HVA`

Confronto appaiato fra MLP diretto, cella physical-τ e cella direct-z sotto voltage clamp costante. Il test sigillato viene aperto soltanto dopo il congelamento della selezione development.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
WORKSPACE = Path('/kaggle/working/giada_task_1')
GIADA_REPO = WORKSPACE / 'giada'
TEACHER_REPO = WORKSPACE / 'neuron_as_deep_net'
OUTPUT_DIR = Path('/kaggle/working/artifacts/giada_gate_m_atomic_playground')
def run(args, cwd=None):
    print('+', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

In [ ]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
if not GIADA_REPO.exists():
    run(['git','clone','--branch','codex/surrogate-validity-audit','--single-branch','https://github.com/Zagred47/giada.git',GIADA_REPO])
else:
    run(['git','fetch','origin','codex/surrogate-validity-audit'], cwd=GIADA_REPO)
    run(['git','checkout','--detach','FETCH_HEAD'], cwd=GIADA_REPO)
if not TEACHER_REPO.exists():
    run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',TEACHER_REPO])
run(['git','checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'], cwd=TEACHER_REPO)
REVISION = subprocess.check_output(['git','rev-parse','HEAD'], cwd=GIADA_REPO, text=True).strip()
print({'giada_revision': REVISION, 'teacher_revision': '074c4666300a8ad246601dab179a97a6942f0f29'})

In [ ]:
sys.path.insert(0, str(GIADA_REPO))
import torch
assert torch.cuda.is_available(), 'La Task 1 registrata richiede una GPU CUDA Kaggle.'
from src.giada_teacher import (AtomicGateTaskConfig, ExtractedGateFormula, materialize_atomic_gate_dataset, train_and_select_atomic_gate, evaluate_frozen_atomic_gate)
prereg = json.loads((GIADA_REPO/'experiments/teacher_gate_m_playground_preregistration_v1.json').read_text())
baseline = json.loads((GIADA_REPO/'experiments/teacher_common_gpu_baseline_v1.json').read_text())
assert baseline['validation']['valid'] and prereg['fairness']['sealed_test_opened_after_freeze']
display({'gpu': torch.cuda.get_device_name(0), 'task': prereg['name'], 'primary_gate': prereg['registered_primary_gate']})

In [ ]:
assert not OUTPUT_DIR.exists(), f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula = ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
dataset = materialize_atomic_gate_dataset(formula, gate='m')
config = AtomicGateTaskConfig()
display({'fit_cases': len(dataset['strata']['train']['inputs']), 'total_cases': sum(len(x['inputs']) for x in dataset['strata'].values()), 'learned_arms': ['direct_mlp','physical_tau','direct_z']})

In [ ]:
training_report = train_and_select_atomic_gate(dataset, OUTPUT_DIR, config)
freeze = json.loads((OUTPUT_DIR/'selection_freeze.json').read_text())
display({'valid': training_report['valid'], 'device': training_report['device'], 'sealed_test_accessed': training_report['sealed_test_accessed'], 'selection': freeze['selection'], 'freeze_sha256': freeze['freeze_sha256']})

In [ ]:
final_report = evaluate_frozen_atomic_gate(dataset, OUTPUT_DIR, config, formula=formula)
display({'valid': final_report['valid'], 'aggregate_sealed_rmse': final_report['aggregate_sealed_rmse'], 'rollout_rmse': final_report['constant_voltage_rollout_rmse'], 'selection_used_sealed_test': final_report['selection_used_sealed_test'], 'claim_scope': final_report['claim_scope']})
assert final_report['valid'] and not final_report['selection_used_sealed_test']

In [ ]:
gate = prereg['registered_primary_gate']
learned = {k:v for k,v in final_report['aggregate_sealed_rmse'].items() if k in {'direct_mlp','physical_tau','direct_z'}}
decision = {
 'any_learned_pass': min(learned.values()) <= gate['any_learned_aggregate_sealed_rmse_max'],
 'physical_tau_pass': final_report['aggregate_sealed_rmse']['physical_tau'] <= gate['physical_tau_aggregate_sealed_rmse_max'],
 'physical_tau_rollout_pass': final_report['constant_voltage_rollout_rmse']['physical_tau']['1000'] <= gate['physical_tau_1000_step_rollout_rmse_max'],
}
decision['task1_gate_passed'] = all(decision.values())
(OUTPUT_DIR/'registered_decision.json').write_text(json.dumps(decision, indent=2))
display(decision)

## 📦 Download robusto
La cella seguente usa il metodo Blob/base64 compatibile con Kaggle già validato nel progetto.

In [ ]:
from base64 import b64encode
from IPython.display import Javascript, display
archive = Path(shutil.make_archive('/kaggle/working/giada_gate_m_atomic_playground', 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = b64encode(archive.read_bytes()).decode('ascii')
filename = archive.name
display(Javascript(f"""
const raw = atob('{payload}');
const bytes = new Uint8Array(raw.length);
for (let i = 0; i < raw.length; i++) bytes[i] = raw.charCodeAt(i);
const url = URL.createObjectURL(new Blob([bytes], {{type: 'application/zip'}}));
const a = document.createElement('a'); a.href = url; a.download = '{filename}';
document.body.appendChild(a); a.click(); a.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'archive': str(archive), 'size_mib': round(archive.stat().st_size/2**20, 2)})